# Matrix Reconstruction (Load Trained Model)
Load a saved Linear AE or AE from disk/Drive, recreate the model from the results JSON, analyze latent distributions, and compute reconstruction metrics (MSE, MAE, Frobenius).

In [46]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Step 1: Load the desired dataset (test.pt / all.pt)
Load correlation matrices.

In [47]:
FILE_NAME = 'data_00_20'
WINDOW_SIZE = 724
STRIDE = 10
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_SIZE}_s{STRIDE}'
RUN = 'AE_100dim_cholesky_02_'
RESULTS_JSON_PATH = f'models/{DATASET_NAME}/AE/{RUN}/run_results.json'


if 'google.colab' in sys.modules:
    print('Environment: Google Colab')
    IS_COLAB = True
else:
    print('Environment: Local PC')
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT = Path('/content/drive/MyDrive')
    base_dir = DRIVE_ROOT / 'dataset_tesi'
else:
    project_root = Path.cwd().resolve().parent
    DRIVE_ROOT = None
    base_dir = project_root / 'data' / 'processed' / FILE_NAME / 'dataset' / DATASET_NAME

dataset_dir = base_dir

DATASET_ORDER = ['test', 'train']
DATASET_FILES = {}
for name in DATASET_ORDER:
    candidate = dataset_dir / f'{name}.pt'
    if candidate.exists():
        DATASET_FILES[name] = candidate

if not DATASET_FILES:
    raise FileNotFoundError(
        f'No dataset .pt files found in: {dataset_dir.absolute()}'
    )

print(f'Dataset selected: {DATASET_NAME}')
print(f'Available datasets: {", ".join(DATASET_FILES.keys())}')

Environment: Local PC
Dataset selected: data_00_20_w724_s10
Available datasets: test, train


In [48]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')

    if isinstance(payload, dict):
        corr_tensor = payload.get('corr_tensor', None)
        chol_tensor = payload.get('chol_tensor', None) # AGGIUNTO
        indices = payload.get('indices', None)
        meta = {k: v for k, v in payload.items() if k not in ('corr_tensor', 'chol_tensor')}
    elif isinstance(payload, torch.Tensor):
        corr_tensor = payload
        chol_tensor = None
        indices = None
        meta = {}
    else:
        raise TypeError(f'Unsupported .pt format: {type(payload)}')

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
        
    if chol_tensor is None:
        raise KeyError('chol_tensor key not found in .pt file. Usa il dataset aggiornato.')

    # Ritorniamo i 4 elementi
    return corr_tensor.float(), chol_tensor.float(), indices, meta

SAMPLE_DATASET = next(iter(DATASET_FILES))

# --- MODIFICA QUI: Aggiunto sample_chol ---
sample_corr, sample_chol, sample_indices, sample_meta = load_corr_payload(DATASET_FILES[SAMPLE_DATASET])

print(f'Sample dataset: {SAMPLE_DATASET}')
print(f'Correlation tensor shape: {sample_corr.shape}')
print(f'Cholesky tensor shape: {sample_chol.shape}') # Utile per verificare!

if sample_indices is not None:
    print(f'Indices: {sample_indices[:10]}')
else:
    print('Indices: not found in payload')

Sample dataset: test
Correlation tensor shape: torch.Size([42, 362, 362])
Cholesky tensor shape: torch.Size([42, 362, 362])
Indices: [279, 23, 272, 15, 316, 109, 137, 190, 287, 16]


## Step 2: Prepare Matrices (Full Dataset)
Use full correlation matrices as input. Each matrix is flattened to a vector of size `N x N`.

In [49]:
def prepare_inputs(chol_tensor: torch.Tensor):
    # Lavoriamo con Cholesky!
    all_chol_np = chol_tensor.numpy().astype(np.float32)
    n_matrices, n_assets, _ = all_chol_np.shape

    # Formula per Cholesky con diagonale
    n_features = n_assets * (n_assets + 1) // 2
    
    # k=0 per includere la diagonale
    tril_idx = np.tril_indices(n_assets, k=0)
    
    extracted_np = all_chol_np[:, tril_idx[0], tril_idx[1]]
    x_all = torch.from_numpy(extracted_np)

    return all_chol_np, x_all, n_assets, n_features, tril_idx

sample_all_np, sample_x_all, N_ASSETS, N_FEATURES, SAMPLE_TRIL_IDX = prepare_inputs(sample_chol)

print(f"{'='*40}")
print(f"Number of matrices   : {sample_all_np.shape[0]}")
print(f"Original matrix shape: ({N_ASSETS}, {N_ASSETS})")
print(f"Flattened input size : {N_FEATURES}")
print(f"Sample tensor shape  : {tuple(sample_x_all.shape)}")
print(f"{'='*40}")

Number of matrices   : 42
Original matrix shape: (362, 362)
Flattened input size : 65703
Sample tensor shape  : (42, 65703)


## Step 3: Define the Models
Use a Linear Autoencoder (no activations) or a non-linear AutoEncoder (with hidden layers).

In [50]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=None, dropout_prob: float = 0.02):
        super().__init__()

        # Se hidden_dims è None, creiamo un Linear AE semplice
        if hidden_dims is None:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, latent_dim, bias=False),
            )
            self.decoder = nn.Sequential(
                nn.Linear(latent_dim, input_dim, bias=False),
            )
        else:
            # Caso Deep Autoencoder con Dropout e attivazioni
            dimensions = [input_dim, *hidden_dims, latent_dim]

            # --- ENCODER ---
            encoder_layers = []
            for i in range(len(dimensions) - 1):
                encoder_layers.append(nn.Linear(dimensions[i], dimensions[i + 1]))
                if i < len(dimensions) - 2:
                    encoder_layers.append(nn.LeakyReLU(0.01))
                    encoder_layers.append(nn.Dropout(dropout_prob))
            self.encoder = nn.Sequential(*encoder_layers)

            # --- DECODER ---
            decoder_dims = dimensions[::-1]
            decoder_layers = []
            for i in range(len(decoder_dims) - 1):
                decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
                if i < len(decoder_dims) - 2:
                    decoder_layers.append(nn.LeakyReLU(0.01))
                #else:
                    #decoder_layers.append(nn.Tanh())
                    
            self.decoder = nn.Sequential(*decoder_layers)

    def architecture_signature(self):
        return [
            [int(layer.in_features), int(layer.out_features)]
            for layer in self.encoder
            if isinstance(layer, nn.Linear)
        ]

    def forward(self, x: torch.Tensor):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

## Step 4: Load Best Model from Results JSON
Select the results JSON path, load model details, recreate the architecture, and load weights.

In [51]:
def resolve_path(path_str, base_dir=None, results_dir=None):
    if path_str is None:
        return None
    p = Path(path_str)
    if p.is_absolute() and p.exists():
        return p.resolve()

    if results_dir is not None:
        candidate = results_dir / p
        if candidate.exists():
            return candidate.resolve()

    if base_dir is not None:
        candidate = base_dir / p
        if candidate.exists():
            return candidate.resolve()

    normalized = str(path_str).replace('\\', '/')
    if base_dir is not None and 'best_models_tesi' in normalized:
        rel = normalized[normalized.index('best_models_tesi'):]
        candidate = base_dir / rel
        if candidate.exists():
            return candidate.resolve()

    return p


results_path = Path(RESULTS_JSON_PATH)
if not results_path.is_absolute():
    if IS_COLAB:
        if DRIVE_ROOT is None:
            raise ValueError('DRIVE_ROOT is not set for Colab')
        results_path = (DRIVE_ROOT / results_path).resolve()
    else:
        results_path = (project_root / results_path).resolve()

if not results_path.exists():
    raise FileNotFoundError(f'Results JSON not found: {results_path}')

OUTPUT_DIR_NAME = 'analysis_outputs'
analysis_dir = results_path.parent / OUTPUT_DIR_NAME
analysis_dir.mkdir(parents=True, exist_ok=True)

with open(results_path, 'r', encoding='utf-8') as f:
    results = json.load(f)

model_cfg = results.get('model', {})
latent_dim = int(model_cfg.get('latent_dim', 0))
if latent_dim <= 0:
    raise ValueError('Invalid latent_dim in results JSON')

hidden_dims = model_cfg.get('hidden_dims', None)
model_type_raw = str(model_cfg.get('model_type', '')).strip().lower()
if model_type_raw in {'linear', 'linearae', 'linear_ae', 'linear-ae'}:
    model_type = 'linearAE'
elif model_type_raw in {'ae', 'autoencoder', 'auto'}:
    model_type = 'AE'
else:
    model_type = 'AE' if hidden_dims is not None else 'linearAE'

if model_type == 'linearAE':
    hidden_dims = None
else:
    if hidden_dims is None:
        raise ValueError('hidden_dims missing for AE in results JSON')

input_dim = int(model_cfg.get('input_dim', N_FEATURES))
if input_dim != N_FEATURES:
    raise ValueError(f'Input dim mismatch: json={input_dim}, data={N_FEATURES}')

dropout_prob = model_cfg.get('dropout', 0.02)
if dropout_prob is None:
    dropout_prob = 0.0
dropout_prob = float(dropout_prob)

model = AutoEncoder(
    input_dim=input_dim,
    latent_dim=latent_dim,
    hidden_dims=hidden_dims,
    dropout_prob=dropout_prob,
).to(device)

weights_path = results.get('artifacts', {}).get('best_model_path', None)

if weights_path is None:
    run_name = results.get('run', 'run')
    if model_type == 'linearAE':
        weights_path = f'linear_AE_best_{run_name}.pt'
    else:
        weights_path = f'AE_best_{run_name}.pt'

base_dir = DRIVE_ROOT if IS_COLAB else project_root
weights_path = resolve_path(weights_path, base_dir=base_dir, results_dir=results_path.parent)
if weights_path is None or not weights_path.exists():
    raise FileNotFoundError(f'Weights file not found: {weights_path}')

checkpoint = torch.load(weights_path, map_location=device)
state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

# --- INIZIO AGGIUNTA: Caricamento statistiche di normalizzazione ---
stats_dir = weights_path.parent
train_mean_path = stats_dir / 'train_mean.pt'
train_std_path = stats_dir / 'train_std.pt'

if not train_mean_path.exists() or not train_std_path.exists():
    raise FileNotFoundError(f"File di normalizzazione mancanti in: {stats_dir}. Assicurati di averli copiati.")

# Carichiamo i tensori e assicuriamoci che siano sulla CPU
train_mean = torch.load(train_mean_path, map_location='cpu')
train_std = torch.load(train_std_path, map_location='cpu')

# Creiamo subito anche le copie NumPy per velocizzare la denormalizzazione
train_mean_np = train_mean.numpy()
train_std_np = train_std.numpy()
# --- FINE AGGIUNTA ---

print(f'Model type: {model_type} | latent_dim={latent_dim} | input_dim={input_dim}')
print(f'Loaded weights: {weights_path}')
print(f'Loaded norm stats: {stats_dir.name}')

Model type: AE | latent_dim=100 | input_dim=65703
Loaded weights: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\best_model.pt
Loaded norm stats: AE_100dim_cholesky_02_


In [52]:
print(model)

AutoEncoder(
  (encoder): Sequential(
    (0): Linear(in_features=65703, out_features=2048, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Dropout(p=0.0, inplace=False)
    (3): Linear(in_features=2048, out_features=1024, bias=True)
    (4): LeakyReLU(negative_slope=0.01)
    (5): Dropout(p=0.0, inplace=False)
    (6): Linear(in_features=1024, out_features=512, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Dropout(p=0.0, inplace=False)
    (9): Linear(in_features=512, out_features=100, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=100, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=1024, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=1024, out_features=2048, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=2048, out_features=65703, bias=True)
  )
)


## Step 5: Latent Space Analysis + Reconstruction Errors Analysis
Encode the matrices into the latent space and analyze feature distributions + Analyse MSE, MAE and Frobenius.

In [53]:
# 1. Funzione di ricostruzione con gestione automatica del device
def compute_reconstruction(model: nn.Module, x_tensor: torch.Tensor, batch_size: int = 64, device=None):
    if device is None:
        device = next(model.parameters()).device
        
    model.eval()
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)
    latents = []
    reconstructions = []

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(device)
            z = model.encoder(xb)
            latents.append(z.cpu().numpy())
            recon = model.decoder(z)
            reconstructions.append(recon.cpu().numpy())

    return np.concatenate(latents, axis=0), np.concatenate(reconstructions, axis=0)

# 2. Funzione per gli errori (rimane invariata, ora riceverà matrici quadrate corrette)
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    if original.shape != reconstructed.shape:
        raise ValueError(f'Input shapes mismatch: {original.shape} vs {reconstructed.shape}')

    diff = original - reconstructed

    
    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [54]:
# ==========================================
# ESECUZIONE DEL CALCOLO E RICOSTRUZIONE
# ==========================================
def process_dataset(dataset_name: str, matrix_file: Path):
    # 1. Caricamento separato
    corr, chol, indices, meta = load_corr_payload(matrix_file)
    all_corr_np = corr.numpy().astype(np.float32) # Ground truth per gli errori
    
    # 2. Preparazione input basato su Cholesky
    _, x_all, n_assets, n_features, tril_idx = prepare_inputs(chol)

    if n_features != N_FEATURES:
        raise ValueError(f'Input dim mismatch for {dataset_name}: expected {N_FEATURES}, got {n_features}')

    # 3. NORMALIZZAZIONE (Z-Score)
    x_all_norm = (x_all - train_mean) / train_std

    # 4. RICOSTRUZIONE
    latents_all, recon_flat_norm = compute_reconstruction(model, x_all_norm, batch_size=64)

    # 5. DENORMALIZZAZIONE (Torniamo nello spazio dei fattori Cholesky)
    recon_flat = (recon_flat_norm * train_std_np) + train_mean_np

    # --- INIZIO NUOVA RICOSTRUZIONE CHOLESKY ---
    n_matrices = all_corr_np.shape[0]
    recon_L = np.zeros((n_matrices, n_assets, n_assets), dtype=np.float32)
    
    # Ricostruiamo la matrice L
    recon_L[:, tril_idx[0], tril_idx[1]] = recon_flat
    
    # Calcolo PSD (C = L * L^T)
    recon_PSD = recon_L @ np.transpose(recon_L, axes=(0, 2, 1))
    
    # Normalizzazione a correlazione
    diag = np.diagonal(recon_PSD, axis1=1, axis2=2)
    d_inv_sqrt = 1.0 / np.sqrt(np.maximum(diag, 1e-12))
    
    recon_all = recon_PSD * d_inv_sqrt[:, :, np.newaxis] * d_inv_sqrt[:, np.newaxis, :]
    recon_all = np.clip(recon_all, -1.0, 1.0)
    
    diag_idx = np.arange(n_assets)
    recon_all[:, diag_idx, diag_idx] = 1.0
    # --- FINE NUOVA RICOSTRUZIONE ---

    # Calcolo errori confrontando con la Correlazione Originale (Ground Truth)
    errors_df, summary_df = reconstruction_errors(all_corr_np, recon_all)

    latent_dim = latents_all.shape[1]
    latent_cols = [f'z{i + 1}' for i in range(latent_dim)]
    latent_df = pd.DataFrame(latents_all, columns=latent_cols)

    if indices is None:
        indices = np.arange(len(errors_df))

    if len(indices) != len(errors_df):
        raise ValueError(f'Error rows ({len(errors_df)}) do not match indices ({len(indices)})')
    errors_df.insert(0, 'matrix_idx', indices)

    print(f'\nReconstruction error summary ({dataset_name} data):')
    display(summary_df)

    errors_json_path = analysis_dir / f'reconstruction_errors_{dataset_name}_{RUN}.json'
    summary_json_path = analysis_dir / f'reconstruction_summary_{dataset_name}_{RUN}.json'
    errors_df.to_json(errors_json_path, orient='records', indent=2)
    summary_df.to_json(summary_json_path, orient='records', indent=2)

    print(f'Saved per-matrix errors: {errors_json_path}')
    print(f'Saved summary stats: {summary_json_path}')

    original_payload = torch.load(matrix_file, map_location='cpu')
    if isinstance(original_payload, dict):
        extended_payload = original_payload.copy()
    else:
        extended_payload = {'corr_tensor': original_payload}

    # Salviamo la correlazione ricostruita
    recon_tensor = torch.from_numpy(recon_all).float()
    extended_payload['corr_tensor_reconstructed'] = recon_tensor
    
    # Opzionale: Salviamo anche i fattori latenti L se servono in futuro
    recon_L_tensor = torch.from_numpy(recon_L).float()
    extended_payload['chol_tensor_reconstructed'] = recon_L_tensor

    tickers = None
    if isinstance(meta, dict):
        tickers = meta.get('meta', {}).get('tickers', None)
    if tickers is not None:
        extended_payload['tickers'] = tickers

    output_path = analysis_dir / f'{dataset_name}_reconstructed_{RUN}.pt'
    torch.save(extended_payload, output_path)
    print(f'Saved reconstructed matrices: {output_path}')

    print(f'\nLatent distribution summary ({dataset_name} data):')
    display(latent_df.describe().T)

    valid_cols = [
        col for col in latent_cols
        if latent_df[col].notna().any() and latent_df[col].nunique() > 1
    ]
    latent_df = latent_df[valid_cols]

    if len(indices) != len(latent_df):
        raise ValueError(f'Latent rows ({len(latent_df)}) do not match indices ({len(indices)})')
    latent_df.insert(0, 'matrix_idx', indices)

    latent_json_path = analysis_dir / f'latent_{dataset_name}_{RUN}.json'
    latent_df.to_json(latent_json_path, orient='records', indent=2)
    print(f'Saved latent samples: {latent_json_path}')

    if len(valid_cols) < 2:
        print('Not enough valid latent dimensions for pairwise plots.')
    elif len(valid_cols) > 3:
        print(f'Skipping pairwise plot: too many latent dimensions ({len(valid_cols)} > 3).')
    else:
        plot_df = latent_df[valid_cols]
        grid = sns.PairGrid(plot_df, corner=True, diag_sharey=False)
        grid.map_lower(sns.scatterplot, s=12, alpha=0.6, color='#2a9d8f')
        grid.figure.suptitle(
            f'Pairwise latent dimension plots ({dataset_name} data)',
            y=1.02,
        )
        pairplot_path = analysis_dir / f'latent_pairwise_{dataset_name}_{RUN}.png'
        grid.savefig(pairplot_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved latent pairwise plot: {pairplot_path}')


for dataset_name, matrix_file in DATASET_FILES.items():
    print(f'\n=== Processing {dataset_name} ({matrix_file.name}) ===')
    process_dataset(dataset_name, matrix_file)


=== Processing test (test.pt) ===

Reconstruction error summary (test data):


,mean,std,min,median,max
MSE,0.000073,0.000081,0.000002,0.000062,0.000464
MAE,0.005113,0.002742,0.001058,0.005394,0.016111
Frobenius,2.750059,1.451076,0.569279,2.852412,7.800651


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\reconstruction_errors_test_AE_100dim_cholesky_02_.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\reconstruction_summary_test_AE_100dim_cholesky_02_.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\test_reconstructed_AE_100dim_cholesky_02_.pt

Latent distribution summary (test data):


,count,mean,std,min,25%,50%,75%,max
z1,42.0,-15.942640,51.642281,-104.797813,-46.639454,-19.387232,11.658212,83.040276
z2,42.0,-11.074014,39.249580,-77.799713,-37.112056,-22.013885,6.664717,65.346397
z3,42.0,7.803207,57.595833,-101.246857,-51.431529,16.869120,57.993191,87.174828
z4,42.0,2.402689,70.138107,-143.612244,-43.350149,-14.318729,43.130348,155.947083
z5,42.0,-9.211868,44.089668,-98.648125,-38.600215,-4.804057,21.656066,82.101730
...,...,...,...,...,...,...,...,...
z96,42.0,3.828547,72.212006,-149.740280,-47.074610,9.601177,38.513687,128.489075
z97,42.0,2.539851,37.493248,-72.104973,-25.429121,4.883590,22.323886,83.448769
z98,42.0,-4.895949,66.021301,-148.047699,-48.189783,-0.634404,30.160001,110.650978
z99,42.0,3.708600,50.392891,-93.980118,-26.707812,9.830995,33.918851,114.168037


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\latent_test_AE_100dim_cholesky_02_.json
Skipping pairwise plot: too many latent dimensions (100 > 3).

=== Processing train (train.pt) ===

Reconstruction error summary (train data):


,mean,std,min,median,max
MSE,8.827329e-07,0.000002,2.669457e-08,3.184632e-07,0.000025
MAE,5.022085e-04,0.000304,1.260221e-04,4.241921e-04,0.001812
Frobenius,2.613455e-01,0.218038,5.914527e-02,2.042849e-01,1.823554


Saved per-matrix errors: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\reconstruction_errors_train_AE_100dim_cholesky_02_.json
Saved summary stats: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\reconstruction_summary_train_AE_100dim_cholesky_02_.json
Saved reconstructed matrices: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\train_reconstructed_AE_100dim_cholesky_02_.pt

Latent distribution summary (train data):


,count,mean,std,min,25%,50%,75%,max
z1,288.0,-17.147200,54.596432,-126.509583,-50.934654,-16.038318,15.387686,91.104042
z2,288.0,-11.375737,38.600613,-78.596573,-39.028081,-15.455255,13.184113,69.216217
z3,288.0,-6.363101,64.334808,-131.606888,-55.462673,0.085556,47.194118,108.977852
z4,288.0,8.175333,77.267052,-154.652893,-34.957271,-1.446500,56.284058,169.436356
z5,288.0,-2.452708,55.823231,-98.581627,-45.965320,-11.996846,41.738438,109.414680
...,...,...,...,...,...,...,...,...
z96,288.0,0.199055,67.945877,-150.600861,-50.940347,7.152429,43.544522,132.046722
z97,288.0,-3.606233,40.102390,-95.435219,-26.539548,-2.965051,22.340152,85.339706
z98,288.0,-6.026678,63.216293,-147.658020,-52.453945,-4.225692,40.872452,110.910988
z99,288.0,-6.636996,52.716290,-112.872406,-49.026889,-4.485330,37.685770,114.281036


Saved latent samples: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w724_s10\AE\AE_100dim_cholesky_02_\analysis_outputs\latent_train_AE_100dim_cholesky_02_.json
Skipping pairwise plot: too many latent dimensions (100 > 3).
